# Notebook 16: PyTorch Basics
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Create and manipulate PyTorch tensors
- Understand automatic differentiation (autograd)
- Build a neural network using `nn.Module`
- Write a complete training loop
- Use `Dataset` and `DataLoader` for batched training

## 1. Why PyTorch?

- **Dynamic computation graph**: builds the graph on-the-fly → easy debugging
- **Autograd**: automatic gradient computation for any tensor expression
- **GPU support**: move tensors with `.to(device)` — same code runs on CPU or GPU
- **Pythonic**: feels like NumPy, integrates with the whole Python ecosystem
- **Ecosystem**: HuggingFace, torchvision, Lightning all built on top

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch version: {torch.__version__}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device  : {device}')

## 2. Tensors

In [ ]:
# Creating tensors
t1 = torch.tensor([1.0, 2.0, 3.0])
t2 = torch.zeros(3, 4)
t3 = torch.randn(2, 3)
t4 = torch.arange(0, 10, 2, dtype=torch.float32)

print('t1:', t1)
print('t2 shape:', t2.shape)
print('t3:\n', t3)
print('t4:', t4)

# Interoperability with NumPy
np_arr = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(np_arr)
print('\nFrom NumPy:', t_from_np)
print('Back to NumPy:', t1.numpy())

In [ ]:
# Operations
a = torch.tensor([[1.,2.],[3.,4.]])
b = torch.tensor([[5.,6.],[7.,8.]])
print('Element-wise *:\n', a * b)
print('Matrix mult @:\n', a @ b)
print('Mean:', a.mean(), ' Sum:', a.sum())

# Broadcasting
c = torch.tensor([10., 20.])
print('Broadcast + c (adds to each row):\n', a + c)

## 3. Autograd — Automatic Differentiation

In [ ]:
# Simple: y = 3x² + 2x + 1, dy/dx = 6x + 2
x = torch.tensor(2.0, requires_grad=True)
y = 3*x**2 + 2*x + 1
y.backward()
print(f'x={x.item()}, y={y.item()}, dy/dx={x.grad.item()}  (expected {6*2+2})')

In [ ]:
# Multi-parameter gradient
torch.manual_seed(42)
w = torch.randn(3, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

X_d = torch.randn(100, 3)
y_d = X_d @ torch.tensor([1.0, -2.0, 0.5]) + 0.3

y_pred = X_d @ w + b
loss = ((y_pred - y_d)**2).mean()
loss.backward()

print(f'Loss: {loss.item():.4f}')
print(f'dL/dw: {w.grad}')
print(f'dL/db: {b.grad}')

## 4. Building a Network with nn.Module

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x): return self.net(x)

model = MLP(8, 64, 1)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. The Training Loop

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X_np = StandardScaler().fit_transform(cancer.data.astype(np.float32))
y_np = cancer.target.astype(np.float32)

X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_tr);  y_train_t = torch.tensor(y_tr).unsqueeze(1)
X_test_t  = torch.tensor(X_te);  y_test_t  = torch.tensor(y_te).unsqueeze(1)

train_dl = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)

model = MLP(30, 64, 1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_losses, val_losses = [], []
for epoch in range(100):
    model.train()
    ep_loss = 0
    for Xb, yb in train_dl:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()
    train_losses.append(ep_loss / len(train_dl))

    model.eval()
    with torch.no_grad():
        v = criterion(model(X_test_t.to(device)), y_test_t.to(device)).item()
    val_losses.append(v)

model.eval()
with torch.no_grad():
    preds = (torch.sigmoid(model(X_test_t.to(device))) > 0.5).float()
    acc = (preds == y_test_t.to(device)).float().mean()
print(f'Test Accuracy: {acc.item():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train')
ax.plot(val_losses,   label='Validation')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.set_title('Training and Validation Loss')
ax.legend()
plt.tight_layout(); plt.show()

## 6. Regularisation Techniques

In [ ]:
class BetterMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)

bm = BetterMLP(30)
print(bm)
print(f'Parameters: {sum(p.numel() for p in bm.parameters()):,}')

## Exercises

1. Add a `StepLR` learning rate scheduler that drops the LR by 0.5 every 30 epochs. Does it improve final accuracy?
2. Implement **early stopping**: stop training if val loss doesn't improve for 10 consecutive epochs.
3. Change the MLP to output 3 classes and train on the Iris dataset. Use `CrossEntropyLoss`.
4. Time training on CPU vs GPU (if available). At what batch size does GPU become faster?

## Reflection Questions
- Why must `optimizer.zero_grad()` be called before each backward pass?
- What is the difference between `BCELoss` and `BCEWithLogitsLoss`? Which is numerically more stable?
- Why do `BatchNorm` and `Dropout` behave differently during training vs evaluation?

---
**Next →** `17_cnn_rnn_lstm.ipynb`